# Routing

Classify an incoming task and send it to the best specialized handler.



## Core ideas



- Intent classification: decide what kind of task arrived before doing the work.
- Specialized handlers: route to prompts, tools, models, agents, or workflows optimized for a case.
- Confidence thresholds: uncertain routes should ask for clarification or use a safe fallback.
- Hierarchical routing: broad category first, narrower route second.
- Evaluation: routing needs confusion-matrix style tests because one wrong route can ruin the task.


## Common failure modes

- Letting ambiguous cases silently fall into the wrong route.
- Using labels that overlap semantically.
- Evaluating only happy-path examples.


## Framework implementation

Routing is the first lesson that benefits from LangGraph: a model produces a typed decision, then a conditional edge selects one specialized node.


In [ ]:
# Import the dependencies used by this example.
import os
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph

# Define the data shape and small operations before running them.
class Route(BaseModel):
    destination: Literal["billing", "technical_support", "general"] = Field(
        description="The single best handler for the request"
    )

class State(TypedDict):
    request: str
    route: str
    response: str

if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: set OPENAI_API_KEY to run model-based routing.")
else:
    router = init_chat_model("openai:gpt-5.6-sol", temperature=0).with_structured_output(Route)

    def classify(state: State):
        # Execute the configured model or workflow with the input below.
        decision = router.invoke(f"Route this customer request: {state['request']}")
        return {"route": decision.destination}

    def billing(state: State): return {"response": "Open the billing workflow."}
    def technical_support(state: State): return {"response": "Start diagnostics."}
    def general(state: State): return {"response": "Ask a clarifying question."}

    # Configure the framework object; this line prepares it but may not execute it yet.
    builder = StateGraph(State)
    builder.add_node("classify", classify)
    builder.add_node("billing", billing)
    builder.add_node("technical_support", technical_support)
    builder.add_node("general", general)
    builder.add_edge(START, "classify")
    builder.add_conditional_edges("classify", lambda state: state["route"])
    for node in ("billing", "technical_support", "general"):
        builder.add_edge(node, END)

    graph = builder.compile()
    print(graph.invoke({"request": "The app crashes while exporting an invoice."})["response"])


## Offline mechanics

This version runs without credentials and exposes the control flow directly.


In [ ]:
# Define the data shape and small operations before running them.
def route(task):
    text = task.lower()
    if any(word in text for word in ["refund", "invoice", "billing"]):
        return "billing"
    if any(word in text for word in ["bug", "error", "crash"]):
        return "technical_support"
    return "general"

handlers = {
    "billing": lambda task: f"Billing agent opens account workflow for: {task}",
    "technical_support": lambda task: f"Support agent starts diagnostics for: {task}",
    "general": lambda task: f"General agent asks a clarifying question about: {task}",
}

task = "The app crashes when I export an invoice"
handlers[route(task)](task)
